# 04 - Pitcher fit, projected results, and free-agent targets

Each pitcher's mix of batted balls (by batter side and type) is weighted by each park's effects. Two different questions come out of that, and they should not be mixed up:

- **Fit** (`fit_pts`): how much better this park suits him than it suits a typical pitcher. It is *relative*. At a very hitter-friendly park every pitcher is hurt, so the best fit is just the pitcher who is hurt least.
- **Projected wOBA allowed** (`proj_woba`): his park-neutral wOBA allowed over the last two seasons (shrunk toward the league average) adjusted by this park's expected effect on him. It combines talent and park, so a lower number means he would actually pitch better there.

Other columns: `edge_pts` is the expected wOBA points on contact vs. a league-average park (positive helps the pitcher), and `talent_woba` is his park-neutral wOBA allowed.

Oakland Coliseum and Steinbrenner Field are left out of scoring because no team plays there now. Sutter Health Park has only 2 seasons of data, so it is kept but low-confidence (see `park_seasons`).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import PROCESSED, INTERIM
from src.park_effects import contact_frame
from src.fit import pitcher_fit, pitcher_talent, project_park_woba, best_pitchers_for_park

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
effects = pd.read_csv(PROCESSED / "park_bb_effects.csv")

names = contact.drop_duplicates("pitcher").set_index("pitcher")["player_name"]
fit = pitcher_fit(contact, effects, min_bip=300)
talent = pitcher_talent(df, effects, recent_seasons=2, shrink_pa=500)
proj = project_park_woba(fit, talent)
proj["park_seasons"] = proj["park"].map(df.groupby("park")["game_year"].nunique())
print(f"{fit['pitcher'].nunique()} pitchers scored for fit, {len(talent)} with a talent estimate, {proj['pitcher'].nunique()} in both")
proj.head()

647 pitchers scored for fit, 710 with a talent estimate, 506 in both


,pitcher,park,coverage,edge_pts,fit_pts,pa,raw_woba,neutral_woba,talent_woba,bip_rate,contact_luck_pts,proj_woba,proj_rank,past_park_pts,regression_pts,new_park_pts,gain_pts,park_seasons
60,434378,American Family Field,1.0,-3.122704,-0.192246,697,0.339311,0.340611,0.331876,0.707317,17.72631,0.334084,389.0,-1.299506,8.735224,-2.208742,5.226976,5
61,434378,Angel Stadium,1.0,-1.023501,0.399020,697,0.339311,0.340611,0.331876,0.707317,17.72631,0.3326,385.0,-1.299506,8.735224,-0.72394,6.711778,5
62,434378,Busch Stadium,1.0,18.054426,2.142525,697,0.339311,0.340611,0.331876,0.707317,17.72631,0.319105,378.0,-1.299506,8.735224,12.770204,20.205921,5
63,434378,Chase Field,1.0,2.151954,1.397935,697,0.339311,0.340611,0.331876,0.707317,17.72631,0.330354,381.0,-1.299506,8.735224,1.522113,8.957831,5
64,434378,Citi Field,1.0,9.062175,-0.075548,697,0.339311,0.340611,0.331876,0.707317,17.72631,0.325466,385.0,-1.299506,8.735224,6.409831,13.845549,5


## Best-fit pitchers for a park

In [2]:
best_pitchers_for_park(fit, "Coors Field", names)

,name,fit_pts,edge_pts
10597,"Little, Brendon",8.869797,-21.730037
9667,"Duran, Jhoan",8.675131,-21.939882
9517,"Graterol, Brusdar",8.246105,-22.388518
9277,"Hill, Tim",8.237575,-22.373869
3757,"Holmes, Clay",8.014637,-22.606431
13057,"Pallante, Andre",7.697795,-22.906537
787,"Alexander, Scott",7.627280,-22.951032
667,"Cobb, Alex",7.023939,-23.589902
7657,"Rogers, Tyler",6.755052,-23.872042
15157,"Fleming, Josh",6.754901,-23.805221


In [3]:
best_pitchers_for_park(fit, "Oracle Park", names)

,name,fit_pts,edge_pts
11925,"Soriano, José",2.538719,5.889500
3765,"Holmes, Clay",2.484453,5.857695
675,"Cobb, Alex",2.387511,5.767979
13065,"Pallante, Andre",2.316909,5.706886
9285,"Hill, Tim",2.270281,5.653146
10605,"Little, Brendon",2.201366,5.595841
9675,"Duran, Jhoan",2.200213,5.579510
18975,"Ureña, Walbert",2.079115,5.345296
17475,"Kochanowicz, Jack",2.036978,5.324892
9105,"Webb, Logan",2.033730,5.401959


## Free-agent class

`data/processed/free_agents_2026.csv` lists this winter's free-agent pitchers (columns `name`, `role`, `team_2026`, `notes`). Names use Statcast's "Last, First" format, and the notebook looks up each pitcher's ID from the data, ignoring accents. Edit the CSV as signings happen.

Pitchers without enough MLB data (for example, players coming from Japan or Korea) will not match and are listed below.

In [4]:
import unicodedata

def key(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return " ".join(s.replace(".", "").replace(",", " ").split())

lookup = {key(n): p for p, n in names.items()}
fa = pd.read_csv(PROCESSED / "free_agents_2026.csv")
fa["mlbam_id"] = fa["name"].map(lambda n: lookup.get(key(n)))
print(f"Matched {fa['mlbam_id'].notna().sum()} of {len(fa)} free agents to Statcast IDs")
print("Not matched:", fa.loc[fa["mlbam_id"].isna(), "name"].tolist())

fa = fa.dropna(subset=["mlbam_id"]).astype({"mlbam_id": int})
fa_proj = proj.merge(fa, left_on="pitcher", right_on="mlbam_id")
print("No score (too few balls in play or recent plate appearances):", sorted(set(fa["name"]) - set(fa_proj["name"])))

Matched 30 of 30 free agents to Statcast IDs
Not matched: []
No score (too few balls in play or recent plate appearances): []


## Who would actually pitch best at each park? (projected wOBA allowed)

The three free agents with the lowest projected wOBA allowed at each park. `park_seasons` shows how much data the park has; treat parks with fewer than 3 seasons as low-confidence.

In [5]:
by_proj = (fa_proj.sort_values("proj_woba").groupby("park").head(3)
                  .sort_values(["park", "proj_woba"]))
cols = ["park", "park_seasons", "name", "role", "proj_woba", "talent_woba", "edge_pts", "fit_pts"]
by_proj[cols].round({"proj_woba": 3, "talent_woba": 3, "edge_pts": 1, "fit_pts": 1}).to_csv(
    PROCESSED / "fa_targets_by_park.csv", index=False)
print(by_proj[cols].round({"proj_woba": 3, "talent_woba": 3, "edge_pts": 1, "fit_pts": 1}).to_string(index=False))

                       park  park_seasons            name role  proj_woba  talent_woba  edge_pts  fit_pts
      American Family Field             5   Skubal, Tarik    S      0.278        0.276      -2.7      0.1
      American Family Field             5 Morejón, Adrian    R      0.282         0.28      -2.3      0.5
      American Family Field             5   Okert, Steven    R      0.283         0.28      -3.6     -0.7
              Angel Stadium             5   Skubal, Tarik    S      0.277        0.276      -1.6     -0.3
              Angel Stadium             5   Okert, Steven    R      0.281         0.28      -0.6      0.8
              Angel Stadium             5 Morejón, Adrian    R      0.282         0.28      -2.1     -0.8
              Busch Stadium             5   Skubal, Tarik    S      0.268        0.276      12.9     -3.2
              Busch Stadium             5   Okert, Steven    R      0.268         0.28      18.5      2.6
              Busch Stadium             5 More

## Buy-low: whose recent results look worse than the pitcher

A pitcher's raw results are distorted by the parks he worked in and by luck on contact. For each pitcher we split the gap between his recent wOBA allowed and his projection into parts (wOBA points, positive = better than his raw results):

- `past_park_pts`: removing the parks he pitched in (a hitter's park inflated his numbers)
- `regression_pts`: pulling extreme results toward the league average
- `new_park_pts`: the new park's expected effect on him (mostly the park-wide effect, which is validated; the pitcher-specific fit is a small extra)

`undervalued_pts` = `past_park_pts` + `regression_pts`: how much his raw results understate his talent, wherever he pitches. `contact_luck_pts` shows how much hitters beat the quality of contact (positive = unlucky for the pitcher; includes defense).

Contract cost is not in this data. Pair the list with a salary projection to find who is cheap relative to his talent.

In [6]:
fa_proj["undervalued_pts"] = fa_proj["past_park_pts"] + fa_proj["regression_pts"]
rnd = {"raw_woba": 3, "neutral_woba": 3, "talent_woba": 3, "best_proj": 3, "undervalued_pts": 1, "contact_luck_pts": 1}

summary = (fa_proj.sort_values("proj_woba")
                  .groupby("name")
                  .agg(role=("role", "first"), pa=("pa", "first"), raw_woba=("raw_woba", "first"),
                       neutral_woba=("neutral_woba", "first"), talent_woba=("talent_woba", "first"),
                       undervalued_pts=("undervalued_pts", "first"), contact_luck_pts=("contact_luck_pts", "first"),
                       best_park=("park", "first"), best_proj=("proj_woba", "first"))
                  .sort_values("undervalued_pts", ascending=False)
                  .reset_index())
summary.round(rnd).to_csv(PROCESSED / "fa_buy_low_summary.csv", index=False)
print(summary.round(rnd).to_string(index=False))

              name role   pa  raw_woba  neutral_woba  talent_woba  undervalued_pts  contact_luck_pts      best_park  best_proj
       May, Dustin    S 1195     0.338          0.34        0.334              4.3               4.1  Busch Stadium      0.323
       Mize, Casey    S 1170      0.33          0.33        0.327              2.5              29.5  Busch Stadium      0.314
   Griffin, Foster    S  715     0.322         0.322        0.321              1.5               1.8  Busch Stadium      0.311
   Peterson, David    S 1371     0.331         0.335        0.331              0.7               2.5 Yankee Stadium      0.322
    Martinez, Nick    S 1390     0.309         0.304        0.308              0.6              -5.5  Busch Stadium      0.295
       Ray, Robbie    S 1446     0.314         0.314        0.315             -1.1              -3.0  Busch Stadium      0.305
       Gray, Sonny    S 1428     0.312         0.312        0.314             -2.6               7.1  Busch Sta

In [7]:
low = (fa_proj.sort_values("gain_pts", ascending=False).groupby("park").head(3)
              .sort_values(["park", "gain_pts"], ascending=[True, False]))
cols = ["park", "park_seasons", "name", "role", "pa", "raw_woba", "proj_woba", "gain_pts",
        "past_park_pts", "regression_pts", "new_park_pts"]
low[cols].round({"raw_woba": 3, "proj_woba": 3, "gain_pts": 1, "past_park_pts": 1, "regression_pts": 1, "new_park_pts": 1}) \
    .to_csv(PROCESSED / "fa_buy_low_by_park.csv", index=False)
print(low[cols].round({"raw_woba": 3, "proj_woba": 3, "gain_pts": 1, "past_park_pts": 1,
                       "regression_pts": 1, "new_park_pts": 1}).to_string(index=False))

                       park  park_seasons            name role   pa  raw_woba  proj_woba  gain_pts  past_park_pts  regression_pts  new_park_pts
      American Family Field             5     May, Dustin    S 1195     0.338      0.336       2.4           -1.8             6.1          -1.9
      American Family Field             5     Mize, Casey    S 1170      0.33      0.329       0.3           -0.6             3.1          -2.2
      American Family Field             5 Griffin, Foster    S  715     0.322      0.323      -0.4            0.7             0.8          -1.9
              Angel Stadium             5     May, Dustin    S 1195     0.338      0.335       3.6           -1.8             6.1          -0.7
              Angel Stadium             5     Mize, Casey    S 1170      0.33      0.328       1.6           -0.6             3.1          -0.9
              Angel Stadium             5 Griffin, Foster    S  715     0.322      0.322       0.6            0.7             0.8       

### Buy-low across the whole league (not just free agents)

Pitchers with 300+ plate appearances in the last two seasons, ranked by how much their raw results understate them. This includes pitchers who are not free agents, so it also works as a trade-target list.

In [8]:
league = (proj.drop_duplicates("pitcher")
              .assign(name=lambda d: d["pitcher"].map(names),
                      undervalued_pts=lambda d: d["past_park_pts"] + d["regression_pts"])
              .query("pa >= 300")
              .sort_values("undervalued_pts", ascending=False)
              .head(40))
out = league[["name", "pa", "raw_woba", "neutral_woba", "talent_woba", "undervalued_pts", "contact_luck_pts"]]
out = out.round({"raw_woba": 3, "neutral_woba": 3, "talent_woba": 3, "undervalued_pts": 1, "contact_luck_pts": 1})
out.to_csv(PROCESSED / "buy_low_league.csv", index=False)
print(out.to_string(index=False))

                name   pa  raw_woba  neutral_woba  talent_woba  undervalued_pts  contact_luck_pts
    Blalock, Bradley  320     0.417         0.405        0.353             63.5               6.3
        Senga, Kodai  506     0.412         0.414        0.367             44.8              34.8
   Rutledge, Jackson  341     0.394         0.397        0.351             43.4              58.7
      Gordon, Colton  443     0.392         0.385         0.35             41.9              47.1
      Gordon, Tanner  802     0.399         0.383        0.359             40.5              28.2
      Woodford, Jake  300     0.382          0.38        0.342             40.1              22.5
     Chivilli, Angel  346     0.383         0.377        0.343             39.9              30.0
       Wilson, Bryse  366     0.388         0.388        0.349             39.8               4.4
       Spence, Mitch  426     0.383         0.373        0.344             39.2               9.3
         Eflin, Zach

## Best park fit, regardless of talent

The three free agents whose profile suits each park best compared with a typical pitcher. This says nothing about how good they are.

In [9]:
by_fit = (fa_proj.sort_values("fit_pts", ascending=False).groupby("park").head(3)
                 .sort_values(["park", "fit_pts"], ascending=[True, False]))
cols = ["park", "park_seasons", "name", "role", "fit_pts", "edge_pts"]
by_fit[cols].round(1).to_csv(PROCESSED / "fa_best_fit_by_park.csv", index=False)
print(by_fit[cols].round(1).to_string(index=False))

                       park  park_seasons               name role  fit_pts  edge_pts
      American Family Field             5       Holmes, Clay    S      1.6      -1.2
      American Family Field             5 Loáisiga, Jonathan    R      1.3      -1.5
      American Family Field             5       Romero, JoJo    R      0.8      -2.0
              Angel Stadium             5      Okert, Steven    R      0.8      -0.6
              Angel Stadium             5      Pivetta, Nick    S      0.6      -0.9
              Angel Stadium             5   Iglesias, Raisel    R      0.5      -1.0
              Busch Stadium             5      Pagán, Emilio    R      5.2      21.0
              Busch Stadium             5        Vesia, Alex    R      3.2      19.1
              Busch Stadium             5      Pivetta, Nick    S      3.1      19.0
                Chase Field             5      Pagán, Emilio    R      3.0       3.7
                Chase Field             5        Vesia, Alex    R

## Best and worst park for each free agent (by fit)

In [10]:
bw = (fa_proj.sort_values("fit_pts")
             .groupby("name")
             .agg(role=("role", "first"), worst_park=("park", "first"), worst=("fit_pts", "first"),
                  best_park=("park", "last"), best=("fit_pts", "last"))
             .round(1)
             .sort_values("best", ascending=False))
print(bw.to_string())

                   role                   worst_park  worst                    best_park  best
name                                                                                          
Holmes, Clay          S             Kauffman Stadium   -6.0           Sutter Health Park   9.0
Loáisiga, Jonathan    R                Busch Stadium   -4.4           Sutter Health Park   6.1
Imanaga, Shota        S           Sutter Health Park   -8.0               loanDepot park   5.5
Peterson, David       S                Busch Stadium   -5.6                Wrigley Field   5.5
Ray, Robbie           S           Sutter Health Park   -6.5  Oriole Park at Camden Yards   5.5
Pagán, Emilio         R           Sutter Health Park   -5.8                Busch Stadium   5.2
Boyd, Matthew         S               Dodger Stadium   -5.0  Oriole Park at Camden Yards   5.1
Rogers, Trevor        S               Dodger Stadium   -4.1  Oriole Park at Camden Yards   4.2
Vesia, Alex           R           Sutter Health Pa

## Next

The backtest in `05_backtest.ipynb` checks whether these park effects predict real results.

## Commit

```
git add notebooks/04_fa_targets.ipynb data/processed
git commit -m "pitcher fit, projected wOBA allowed, free-agent targets"
git push
```